In [18]:
from langgraph.graph import StateGraph,START,END
import os

from dotenv import load_dotenv

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from typing import TypedDict



In [19]:
load_dotenv()

True

In [34]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

load_dotenv()

token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

if not token:
    raise ValueError("Hugging Face token is missing")

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    max_new_tokens=500,
    huggingfacehub_api_token=token,
)

model = ChatHuggingFace(llm=llm)

In [35]:
model = ChatHuggingFace(llm=llm)

In [22]:
class LLMState(TypedDict):

    question: str
    answer : str

In [30]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [32]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()


In [36]:
intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ad60da9-033fcd311889167c41514da2;bfd7c79d-a9da-4aef-a518-29e8b14b99ab)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

In [37]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

load_dotenv()

token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

if not token:
    raise ValueError("Hugging Face token not found in .env")

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    huggingfacehub_api_token=token,
    max_new_tokens=500,
    temperature=0.7,
)

model = ChatHuggingFace(llm=llm)

response = model.invoke("Explain REST APIs in simple words.")
print(response.content)

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ad60dc7-045db436211b55967a607850;e2969c88-c1a7-4f3e-aed8-8f14e091e5ed)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

In [38]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient(
    token=os.getenv("HF_TOKEN")
)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "Reply with exactly: Hello"}
    ],
    max_tokens=20,
)

print(response.choices[0].message.content)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ad60dfb-2a438d0179289a2570e5c7bf;24a87aef-691c-4ae7-8686-dc0c8046c910)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.

In [41]:
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

load_dotenv()

# 1. Initialize the model
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
    max_tokens=500,
)

# 2. Define the graph state
class LLMState(TypedDict):
    question: str
    answer: str

# 3. Create the node
def llm_qa(state: LLMState) -> dict:
    prompt = f"Answer the following question: {state['question']}"

    response = model.invoke(prompt)

    return {
        "answer": response.content
    }

# 4. Build the workflow
graph = StateGraph(LLMState)

graph.add_node("llm_qa", llm_qa)
graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)

workflow = graph.compile()

# 5. Execute the workflow
initial_state = {
    "question": "How far is the Moon from Earth?",
    "answer": "",
}

final_state = workflow.invoke(initial_state)

print("Question:", final_state["question"])
print("Answer:", final_state["answer"])

Question: How far is the Moon from Earth?
Answer: The average distance from the Earth to the Moon is about **384,400 km** (≈ 238,900 miles).  
Because the Moon follows an elliptical orbit, this distance varies:

| Point in orbit | Distance (km) | Distance (mi) |
|----------------|---------------|---------------|
| Perigee (closest) | ~363,300 | ~225,700 |
| Apogee (farthest) | ~405,500 | ~251,900 |

So while the mean separation is ~384 400 km, the Moon can be as close as ~363 000 km or as far as ~406 000 km from Earth.
